In [101]:
import polars as pl

In [102]:
books_df = pl.scan_ndjson("../processed-data/cleaned_books_fantasy_paranormal.json")
interactions_df = pl.scan_ndjson("../processed-data/cleaned_interactions_fantasy_paranormal.json")
authors_df = pl.scan_ndjson("../raw-data/goodreads_book_authors.json")
series_df = pl.scan_ndjson("../raw-data/goodreads_book_series.json")

In [103]:
# filter out descriptions with fewer than 50 words
books_df = books_df.filter(pl.col('description').str.count_matches(r"\w+") >= 50)

In [104]:
books_df.select('language_code').unique().show()

language_code
str
"""zho"""
"""msa"""
"""ale"""
"""en-US"""
"""cat"""


In [105]:
# Keep only english variations
# """en"""
# """en-CA"""
# """en-GB"""
# """en-US"""
# """eng"""

books_df = books_df.filter(
    pl.col('language_code').is_in(['en', 'en-CA', 'en-GB', 'en-US', 'eng'])
)


In [106]:
books_df = books_df.explode("authors").with_columns(
    pl.col("authors").struct.field("author_id").alias("author_id")
)
books_df = books_df.join(
    authors_df.select(["author_id", "name"]),
    on="author_id",
    how="left"
).group_by("book_id").agg(
    pl.all().exclude(["authors", "author_id", "name"]).first(),
    pl.col("name").drop_nulls().alias("author_names")
)


In [107]:
books_df = books_df.explode("series").with_columns(
    pl.col("series").alias("series_id")
)
books_df = books_df.join(
    series_df.select(["series_id", "title"]),
    on="series_id",
    how="left"
).group_by("book_id").agg(
    pl.all().exclude(["series", "series_id", "title"]).first(),
    pl.col("title").drop_nulls().alias("series")
)


In [108]:
books_df = books_df.with_columns(
    pl.col("popular_shelves").list.eval(
        pl.element().struct.field("name")
    ).alias("popular_shelves")
).with_columns(
    pl.col("popular_shelves").list.eval(
        pl.element().filter(
            ~pl.element().str.contains(r"(?i)read|own|buy|fav|library|audio|kindle|ebook")
        )
    )
)


In [109]:
books_df.select('language_code').unique().show()

language_code
str
"""en-CA"""
"""en-GB"""
"""eng"""
"""en"""
"""en-US"""


In [110]:
books_df.head().show()

book_id,isbn,text_reviews_count,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,similar_books,description,format,link,publisher,num_pages,publication_day,isbn13,publication_month,edition_information,publication_year,url,image_url,ratings_count,work_id,title_without_series,author_names,title_right,series
i64,str,i64,str,str,list[str],str,str,f64,str,list[str],str,str,str,str,i64,i64,str,i64,str,i64,str,str,i64,i64,str,list[str],str,list[str]
26868501,"""""",10,"""US""","""en-CA""","[""fantasy"", ""paranormal"", … ""was-free""]","""B0164E933A""","""true""",3.99,"""B0164E933A""","[""7671347"", ""9698150"", … ""19228935""]","""Her identity is secret. Her de…","""""","""https://www.goodreads.com/book…","""""",408,null,"""""",null,"""""",null,"""https://www.goodreads.com/book…","""https://s.gr-assets.com/assets…",75,46911687,"""Tressa's Treasure's (The King'…","[""Belinda M. Gordon""]",null,"[""Tressa's Treasure's (The King's Jewel Book 1)""]"
18769734,"""1781082405""",40,"""US""","""eng""","[""fantasy"", ""magic"", … ""d-g""]","""""","""false""",3.97,"""B00N17UY3C""","[""18668302"", ""22748429"", … ""18053568""]","""The Seven Kingdoms are in the …","""Mass Market Paperback""","""https://www.goodreads.com/book…","""Solaris""",559,27,"""9781781082409""",8,"""""",2014,"""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…",589,26669900,"""The Fire Prince (The Cursed Ki…","[""Emily Gee""]","""The Cursed Kingdoms""","[""The Fire Prince (The Cursed Kingdoms, #2)""]"
35609314,"""1942122640""",14,"""US""","""eng""","[""paranormal-romance"", ""bdsm"", … ""liked-author""]","""""","""false""",4.41,"""B0751FYDYB""",[],"""Loss left them only rage...Unt…","""Paperback""","""https://www.goodreads.com/book…","""Story Witch Press""",502,19,"""9781942122647""",9,"""""",2017,"""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…",61,57048831,"""Vampire's Soul""","[""Joey W. Hill""]","""Vampire Queen""","[""Vampire's Soul""]"
29924516,"""""",3,"""US""","""eng""","[""fantasy"", ""books"", … ""adventure""]","""B01D1ZKSXO""","""true""",3.77,"""B01D1ZKSXO""",[],"""Welcome to a world of muskets …","""""","""https://www.goodreads.com/book…","""""",null,null,"""""",null,"""""",null,"""https://www.goodreads.com/book…","""https://s.gr-assets.com/assets…",13,50307368,"""A Tumultuous Convergence (The …","[""Christopher Kastensmidt""]","""The Elephant and Macaw Banner …","[""A Tumultuous Convergence (The Elephant and Macaw Banner - Novelette Series, #6)""]"
17447009,"""""",3,"""US""","""en-GB""","[""fantasy"", ""short-stories"", … ""amazon""]","""B0045JL5NS""","""true""",3.92,"""B0045JL5NS""",[],"""ODD TALES is an anthology of s…","""Kindle Edition""","""https://www.goodreads.com/book…","""""",45,20,"""""",9,"""""",2010,"""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…",12,24326917,"""Odd Tales""","[""Jeffrey Dean Doty""]",null,"[""Odd Tales""]"


In [111]:
books_df = books_df.drop([
    'isbn', 'text_reviews_count', 'country_code', 'language_code',
    'is_ebook', 'kindle_asin', 'format', 'num_pages',
    'publication_day', 'isbn13', 'publication_month', 'edition_information', 'publication_year', 'image_url',
    'title_without_series', 'ratings_count', 'publisher', 'similar_books', 'book_id', 'asin'
])
books_df.head().show()

popular_shelves,average_rating,description,link,url,work_id,author_names,title_right,series
list[str],f64,str,str,str,i64,list[str],str,list[str]
"[""manga"", ""romance"", … ""01-romance-paranormal""]",4.24,"""He loves her blood, but does h…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",21527396,"[""Kanoko Sakurakouji""]","""Black Bird""","[""Black Bird, Vol. 16 (Black Bird, #16)""]"
"[""paranormal-romance"", ""paranormal"", … ""reviewed""]",3.6,"""Roane leaves the Fury werewolf…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",10961166,"[""Anya Bast""]","""Otherkin""","[""Tranquility (Otherkin, #2)""]"
"[""paranormal-romance"", ""paranormal"", … ""no-thanks-sir-angel""]",4.11,"""In this stunning and sexy addi…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",14816133,"[""Meljean Brook""]","""The Guardians""","[""Demon Marked (The Guardians, #7)""]"
"[""paranormal"", ""new-adult"", … ""continued-series""]",4.1,"""**Mature Content Warning** 17+…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",26518244,"[""Jessica Sorensen""]","""Shattered Promises""","[""Fractured Souls (Shattered Promises, #2)""]"
"[""m-m"", ""mm"", … ""ascavenger-hunt-2015""]",3.63,"""A remote farmhouse in Cornwall…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",45237982,"[""Annabelle Jacobs""]","""Lycanaeris""","[""The Altered 2 (Lycanaeris #2)""]"
